# 推論・評価・比較

[training.ipynb](training.ipynb) と同じEXPERIMENT_IDを指定します。検証済みZIPから完了したBaselineと選択した転移モデルを復元し、共通の生成器とdevセットで評価します。既定ではBaselineだけを評価します。学習済みの転移モデルを比較するときに、対応するCOMPAREスイッチをTrueにします。

`DATA_CONFIG` または入力先のOverrideで、学習時と同じコーパスルートとmanifestを指定します。既存モデルを評価する場合は、その保存先に対応する元の実験IDを指定してください。

In [ ]:
#@title 共通ログ：開始・逐次出力・生存通知・完了・失敗
from pathlib import Path
import json
"""Standard-library-only process logging, also embedded in Colab before install."""


def log_event(name, state, logs, detail=""):
    from datetime import datetime, timezone
    from pathlib import Path
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    line = f"{datetime.now(timezone.utc).isoformat()} [{name}] {state} {detail}\n"
    with (root / "notebook-events.log").open("a", encoding="utf-8") as stream:
        stream.write(line)
        stream.flush()
    print(line, end="", flush=True)


def run_process(command, logs, *, name="command", capture_json=False, env=None, cwd=None,
                heartbeat_seconds=10., extra_record=None):
    """Stream every child line; silence triggers liveness, never fake progress.

    stdout JSON is kept apart from stderr. Nested loggers share an owned POSIX
    process group so interruption cannot strand a second training process.
    """
    from datetime import datetime, timezone
    import hashlib
    import json
    import os
    from pathlib import Path
    import queue
    import signal
    import subprocess
    import tempfile
    import threading
    import time

    if heartbeat_seconds <= 0:
        raise ValueError("heartbeat_seconds must be positive")
    root = Path(logs)
    root.mkdir(parents=True, exist_ok=True)
    command = list(map(str, command))
    key = hashlib.sha256(json.dumps(name, ensure_ascii=False).encode()).hexdigest()[:12] + "-" + str(time.time_ns())
    output, result_path, receipt = root / (key + ".log"), root / (key + ".stdout.log"), root / (key + ".json")
    launch_env = dict(os.environ if env is None else env)
    launch_env["PYTHONUNBUFFERED"] = "1"
    owns_group = os.name == "posix" and launch_env.get("AT_LOG_MANAGED_GROUP") != "1"
    if owns_group:
        launch_env["AT_LOG_MANAGED_GROUP"] = "1"
    started, clock = datetime.now(timezone.utc).isoformat(), time.perf_counter()
    record = dict(name=name, command=command, started_utc=started,
        host_utc_offset_seconds=int(datetime.now().astimezone().utcoffset().total_seconds()),
        elapsed_seconds=0., returncode=None, status="running", exception=None,
        pid=None, log=str(output.resolve()), stdout_log=str(result_path.resolve()) if capture_json else None,
        gpu_csv=None, gpu_monitor_error=None, cwd=str(Path(cwd or Path.cwd()).resolve()))
    lock, stop, done = threading.Lock(), threading.Event(), threading.Event()
    messages, reader_errors = queue.Queue(maxsize=256), []
    process = reader = result_stream = None
    stream = output.open("w", encoding="utf-8")

    def save_record():
        record["elapsed_seconds"] = time.perf_counter() - clock
        fd, temporary = tempfile.mkstemp(dir=root, prefix=".log-record-")
        try:
            with os.fdopen(fd, "w", encoding="utf-8") as target:
                json.dump(record, target, ensure_ascii=False, indent=2, allow_nan=False)
                target.write("\n")
                target.flush()
            os.replace(temporary, receipt)
        finally:
            if os.path.exists(temporary):
                os.unlink(temporary)

    def notice(state, detail=""):
        line = f"[{name}] {state} pid={record['pid']} elapsed={time.perf_counter()-clock:.1f}s {detail}\n"
        with lock:
            stream.write(line)
            stream.flush()
        print(line, end="", flush=True)

    def read_lines(pipe):
        try:
            for line in pipe:
                with lock:
                    stream.write(line)
                    stream.flush()
                while not stop.is_set():
                    try:
                        messages.put(line, timeout=.1)
                        break
                    except queue.Full:
                        pass
        except Exception as error:
            reader_errors.append(error)
        finally:
            done.set()

    def terminate_child():
        if process is None:
            return
        if owns_group:
            try:
                os.killpg(process.pid, signal.SIGTERM)
            except ProcessLookupError:
                pass
        elif process.poll() is None:
            process.terminate()
        try:
            process.wait(timeout=5)
        except subprocess.TimeoutExpired:
            if owns_group:
                os.killpg(process.pid, signal.SIGKILL)
            else:
                process.kill()
            process.wait()
        if owns_group:
            # A nested child may outlive its exited leader after cancellation.
            try:
                os.killpg(process.pid, signal.SIGKILL)
            except ProcessLookupError:
                pass

    try:
        notice("started", f"full log: {output.resolve()}")
        save_record()
        if capture_json:
            result_stream = result_path.open("w", encoding="utf-8")
        process = subprocess.Popen(command, stdout=result_stream if capture_json else subprocess.PIPE,
            stderr=subprocess.PIPE if capture_json else subprocess.STDOUT, text=True,
            encoding="utf-8", errors="replace", bufsize=1, env=launch_env, cwd=cwd,
            start_new_session=owns_group)
        record["pid"] = process.pid
        notice("running", "waiting for child output")
        save_record()
        pipe = process.stderr if capture_json else process.stdout
        reader = threading.Thread(target=read_lines, args=(pipe,), daemon=True, name="command-log-reader")
        reader.start()
        last_output = last_notice = time.perf_counter()
        while not (done.is_set() and messages.empty() and process.poll() is not None):
            try:
                line = messages.get(timeout=min(.1, heartbeat_seconds))
            except queue.Empty:
                line = None
            if line is not None:
                print(line, end="", flush=True)
                if line.strip():
                    last_output = time.perf_counter()
            if reader_errors:
                raise reader_errors[0]
            now = time.perf_counter()
            if now - max(last_output, last_notice) >= heartbeat_seconds:
                notice("running", f"no_output={now-last_output:.1f}s (liveness only; progress not confirmed)")
                save_record()
                last_notice = now
        code = process.wait()
        if reader_errors:
            raise reader_errors[0]
        if capture_json and code == 0:
            json.loads(result_path.read_text(encoding="utf-8"))
        record.update(returncode=code, status="succeeded" if code == 0 else "failed_or_interrupted")
        if code:
            raise subprocess.CalledProcessError(code, command)
        notice("completed", f"exit={code}; full log: {output.resolve()}")
    except BaseException as error:
        record.update(status="failed_or_interrupted", exception=type(error).__name__ + ": " + str(error))
        terminate_child()
        if process is not None:
            record["returncode"] = process.returncode
        notice("interrupted" if isinstance(error, KeyboardInterrupt) else "failed",
               f"exit={record['returncode']} {type(error).__name__}; full log: {output.resolve()}")
        raise
    finally:
        stop.set()
        if reader is not None:
            reader.join(timeout=2)
        if process is not None and (reader is None or not reader.is_alive()):
            pipe = process.stderr if capture_json else process.stdout
            if pipe is not None:
                pipe.close()
        if result_stream is not None:
            result_stream.close()
        if extra_record is not None:
            record.update(extra_record() if callable(extra_record) else extra_record)
        save_record()
        stream.close()
    result = subprocess.CompletedProcess(command, record["returncode"],
        result_path.read_text(encoding="utf-8") if capture_json else None)
    result.record = record
    return result


def execute(*arguments, name=None, capture_json=False, env=None, cwd=None):
    command = list(map(str, arguments))
    if name is None:
        name = command[2].rsplit('.', 1)[-1] if len(command) > 2 and command[1] == '-m' else Path(command[0]).name
        if '--method' in command:
            name += ':' + command[command.index('--method') + 1]
        if '--prior-type' in command:
            name += ':' + command[command.index('--prior-type') + 1]
    return run_process(command, LOGS, name=name, capture_json=capture_json, env=env, cwd=cwd)


def execute_json(*arguments, name=None, env=None, cwd=None):
    return json.loads(execute(*arguments, name=name, capture_json=True, env=env, cwd=cwd).stdout)

LOGS = Path('/content/alignment-transfer-logs/evaluate/bootstrap')

In [ ]:
from google.colab import drive
log_event('drive-mount', 'started', LOGS)
try:
    drive.mount('/content/drive')
except BaseException as error:
    log_event('drive-mount', 'failed', LOGS, type(error).__name__)
    raise
else:
    log_event('drive-mount', 'completed', LOGS)
import json, os, subprocess, sys, time
from pathlib import Path


In [ ]:
#@title 評価する実験と条件
EVALUATE_S2UT = True #@param {type:"boolean"}
EVALUATE_T2TT_EXTERNAL = False #@param {type:"boolean"}
T2TT_RUN_ID = 't2tt-001' #@param {type:"string"}
T2TT_MAX_UPDATES = 200000 #@param {type:"integer"}
T2TT_TOKENIZER = 'sentencepiece' #@param ['character', 'sentencepiece']
REFERENCE_BENCHMARK = '' #@param {type:"string"}
SOURCE_8C = '' #@param {type:"string"}
EXPERIMENT_ID = 'experiment-001' #@param {type:"string"}
DATA_CONFIG = 'configs/data/colab.yaml' #@param {type:"string"}
SOURCE_MANIFEST_OVERRIDE = '' #@param {type:"string"}
RELEASE_SUMMARY_OVERRIDE = '' #@param {type:"string"}
CORPUS_ROOT_OVERRIDE = '' #@param {type:"string"}
COMPARE_DISTILLATION = False #@param {type:"boolean"}
COMPARE_ANNEALED_DISTILLATION = False #@param {type:"boolean"}
COMPARE_MASK_DISTILLATION = False #@param {type:"boolean"}
GENERATE_WAVEFORMS = False #@param {type:"boolean"}
ASR_BLEU = False #@param {type:"boolean"}
if ASR_BLEU and not GENERATE_WAVEFORMS:
    raise ValueError('ASR-BLEUを計算する場合はGENERATE_WAVEFORMS=Trueにしてください')
PREPARE_AUDIO_PACKS = False

In [ ]:
#@title 保存先と固定環境
import os
import sys
for value in (EXPERIMENT_ID, globals().get('T2TT_RUN_ID', 'text')):
    if not value or Path(value).name != value or value in ('.', '..'):
        raise ValueError('実験名は単一のディレクトリ名にしてください')
REPO_URL = 'https://github.com/Yaaamashiro/alignment-transfer-s2ut.git'
REPO_REVISION = '8b8b778b97f626ef04f5aeb87d31541728c003c4'
REPO = Path('/content/alignment-transfer-s2ut')
PYTHON = '/content/alignment-venv/bin/python'
ROOT = str(Path('/content/alignment-transfer-work') / EXPERIMENT_ID)
PERSISTENT = str(Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / EXPERIMENT_ID)
LOGS = Path(ROOT) / 'execution'
BENCHMARK_RUNTIME = Path('/content/alignment-benchmark-runtime')
BENCHMARK_RUNTIME_REVISION = '0c0024ea888206149a3119bf096b86ba66d63b66'
DATA = Path(ROOT) / 'data'
COMMON = DATA / 'common'
BENCHMARK_UNITS = DATA / 's2ut/units'
PREPARED = DATA / 's2ut/fairseq'
BACKUP = Path(PERSISTENT) / 'baseline'
RESTORED = Path(ROOT) / 'baseline'
BASELINE = Path(ROOT) / 'registrations/baseline.json'
MANIFEST = Path(ROOT) / 'manifest/manifest.jsonl'
BACKBONE_CONFIG = Path(ROOT) / 'backbone.json'
SCOPE_SUFFIX = '-spm-all-layers'
RESEARCH_ROOT = Path(ROOT) / ('research-v3' + SCOPE_SUFFIX)
METHOD_LABELS = [
    ('distillation', 'KL-based Attention Distillation'),
    ('annealed_distillation', 'Annealed KL-based Attention Distillation'),
    ('mask_distillation', 'Attention-level Knowledge Distillation')]
os.environ.update(HF_HOME='/content/huggingface', MPLBACKEND='Agg', PYTHONUNBUFFERED='1', PYTHONUTF8='1',
    MPLCONFIGDIR='/content/matplotlib-cache', S2ST_DRIVE_SAFE='1',
    S2ST_BACKUP_MIN_SECONDS='600', S2ST_BACKUP_MAX_GB='1024')


## 1. 環境の構築と検査

In [ ]:
#@title Python・fairseq環境を構築
if not (REPO / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', REPO_URL, str(REPO), name='clone-alignment')
execute('git', '-C', str(REPO), 'fetch', 'origin', REPO_REVISION, name='fetch-alignment')
execute('git', '-C', str(REPO), 'checkout', '--detach', REPO_REVISION, name='checkout-alignment')
os.chdir(REPO)
execute(sys.executable, '-m', 'pip', 'install', 'uv', name='install-uv')
execute('uv', 'venv', '--python', '3.10.22', '--seed', '--allow-existing', '/content/alignment-venv', name='create-python-environment')
for packages in [
    ['pip==23.3.2', 'setuptools<81', 'wheel'],
    ['torch==2.7.1', 'torchaudio==2.7.1', 'numpy==1.26.4', 'Cython==3.2.9'],
    ['--no-build-isolation', '-r', 'requirements-fairseq.txt'],
    ['-r', 'requirements.txt'],
]:
    execute(PYTHON, '-m', 'pip', 'install', *packages)


In [ ]:
#@title benchmark runtimeを固定revisionで導入
if not (BENCHMARK_RUNTIME / '.git').is_dir():
    execute('git', 'clone', '--no-checkout', 'https://github.com/Yaaamashiro/ja-en-direct-s2st-benchmark.git', str(BENCHMARK_RUNTIME))
execute('git', '-C', str(BENCHMARK_RUNTIME), 'fetch', 'origin', BENCHMARK_RUNTIME_REVISION, name='fetch-benchmark-runtime')
execute('git', '-C', str(BENCHMARK_RUNTIME), 'checkout', '--detach', BENCHMARK_RUNTIME_REVISION, name='checkout-benchmark-runtime')
execute(PYTHON, '-m', 'pip', 'install', 'PyYAML==6.0.2', '--no-deps', '-e', str(BENCHMARK_RUNTIME))
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:

config = execute_json(PYTHON, '-c',
    "import json,sys,yaml; print(json.dumps(yaml.safe_load(open(sys.argv[1], encoding='utf-8'))))",
    DATA_CONFIG, name='data-settings')
needs_speech = globals().get('PREPARE_S2UT', globals().get('EVALUATE_S2UT', False))
root_value = CORPUS_ROOT_OVERRIDE or config.get('corpus_root')
manifest_value = SOURCE_MANIFEST_OVERRIDE or config.get('accepted_manifest')
summary_value = RELEASE_SUMMARY_OVERRIDE or config.get('release_summary')
if needs_speech and (not root_value or not manifest_value):
    raise ValueError('DATA_CONFIGまたはCORPUS_ROOT_OVERRIDE / SOURCE_MANIFEST_OVERRIDEでコーパスとmanifestを指定してください')
CORPUS_ROOT = Path(root_value) if root_value else None
SOURCE_MANIFEST = Path(manifest_value) if manifest_value else None
RELEASE_SUMMARY = Path(summary_value) if summary_value else None
if CORPUS_ROOT and any(path.resolve().is_relative_to(CORPUS_ROOT.resolve()) for path in (Path(ROOT), Path(PERSISTENT))):
    raise ValueError('出力先をコーパス原本の外に指定してください')
os.environ.pop('S2ST_INPUT_MAP', None)
execute(PYTHON, '-m', 'pip', 'install', 'editdistance==0.8.1', 'librosa==0.10.2.post1',
    'accelerate==1.12.0', 'pandas==2.2.3', 'psutil==7.2.2', name='runtime-dependencies')
execute(PYTHON, '-m', 'pip', 'check', name='dependency-check')


In [ ]:
execute(PYTHON, '-m', 'alignment_transfer_s2ut.environment', '--native', '--benchmark', '--cuda',
    name='evaluation-environment')

In [ ]:
#@title 実行・保存・復元の共通処理
def corpus_stage(name, module, *arguments, work, restore_only=False):
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run', '--work', str(work),
        '--backup', str(Path(PERSISTENT) / 'stages' / name)]
    if restore_only:
        command.append('--restore-only')
    execute(*command, '--', PYTHON, '-m', module, *map(str, arguments), name=name)

def corpus_common(restore_only=False):
    os.environ['CORPUS_ROOT'] = str(CORPUS_ROOT)
    summary_args = ['--summary', str(RELEASE_SUMMARY)] if RELEASE_SUMMARY else []
    corpus_stage('common', 'alignment_transfer_s2ut.benchmark_data', 'corpus',
        '--source', SOURCE_MANIFEST, '--corpus-root', CORPUS_ROOT, *summary_args,
        '--output', COMMON, '--resume', work=COMMON, restore_only=restore_only)
    lock = json.loads((COMMON / 'dataset-lock.json').read_text(encoding='utf-8'))
    expected = config.get('expected_pairs')
    if expected is not None and lock['total_pairs'] != expected:
        raise ValueError(f"train/dev count {lock['total_pairs']} differs from expected_pairs={expected}")
    print({'pairs': lock['total_pairs'], 'splits': lock['splits'],
        'estimated_waveform_GiB': round(lock['wav_bytes_estimate'] / 1024**3, 1)})

def corpus_stage_audio(create=False):
    staged = execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'stage-corpus',
        '--common', str(COMMON), '--corpus-root', str(CORPUS_ROOT), '--packs', str(Path(PERSISTENT) / 'wav-packs'),
        '--local', str(Path(ROOT) / 'staged-inputs'), *(['--create-audio'] if create else []), name='audio-staging')
    if not create:
        os.environ['S2ST_INPUT_MAP'] = staged['input_map']
    print(staged)

def corpus_prepare(restore_only=False):
    corpus_stage('unit-extraction', 'alignment_transfer_s2ut.corpus_runtime', 'units',
        '--common', COMMON, '--output', BENCHMARK_UNITS, '--cache', '/content/alignment-assets',
        work=BENCHMARK_UNITS, restore_only=restore_only)
    corpus_stage('fairseq-data', 'alignment_transfer_s2ut.corpus_runtime', 'prepare',
        '--common', COMMON, '--units', BENCHMARK_UNITS, '--output', PREPARED,
        work=PREPARED, restore_only=restore_only)

def corpus_baseline(restore_only=False):
    execute(PYTHON, '-m', 'alignment_transfer_s2ut.corpus_runtime', 'baseline',
        '--data', str(DATA), '--work', str(RESTORED), '--backup', str(BACKUP),
        *(['--restore-only'] if restore_only else ['--train']), name='baseline')
def run(name, module, *arguments, gpu=False, env=None, restore_only=False):
    arguments = list(map(str, arguments))
    payload = [PYTHON, '-m', module, *arguments]
    # Read-only environment checks and asset downloads have no stage output.
    if '--output' in arguments and module != 'alignment_transfer_s2ut.benchmark' or (
            module == 'alignment_transfer_s2ut.benchmark' and arguments[0] != 'plan'):
        if '--output' in arguments:
            output = Path(arguments[arguments.index('--output') + 1])
            work = output.parent if output.suffix in ('.json', '.jsonl', '.png') else output
            wrapper = [PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'run',
                '--work', str(work), '--backup', str(Path(PERSISTENT) / 'stages' / name)]
            if restore_only:
                wrapper.append('--restore-only')
            if module == 'alignment_transfer_s2ut.train_t2tt' or module == 'alignment_transfer_s2ut.benchmark' and arguments[0] == 'train':
                wrapper.append('--training')
            payload = [*wrapper, '--', *payload]
    command = [PYTHON, '-m', 'alignment_transfer_s2ut.pilot', 'run', '--name', name, '--logs', str(LOGS)]
    if gpu:
        command.append('--monitor-gpu')
    return execute(*command, '--', *payload, name='notebook:' + name, env=env)

## T2TTの外部BLEU（任意）

EVALUATE_T2TT_EXTERNAL=Trueで、保存済み8Cの固定300件を使って翻訳品質を確認します。T2TTだけを評価する場合はEVALUATE_S2UT=Falseにします。

In [ ]:
if EVALUATE_T2TT_EXTERNAL:
    reference = REFERENCE_BENCHMARK or config.get('benchmark_reference_root')
    if not reference:
        raise ValueError('REFERENCE_BENCHMARKに保存済み外部評価セットの実験ルートを指定してください')
    text_work = Path('/content/alignment-transfer-work') / T2TT_RUN_ID / ('t2tt-' + T2TT_TOKENIZER)
    text_backup = Path('/content/drive/MyDrive/alignment-transfer-s2ut-data') / T2TT_RUN_ID / 'stages' / ('t2tt-' + T2TT_TOKENIZER)
    execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'restore-inputs',
        '--work', str(text_work), '--backup', str(text_backup), '--expected-updates', str(T2TT_MAX_UPDATES),
        name='restore-external-text-model')
    fixed_set = Path(ROOT) / 'external-text/fixed-set'
    source_args = ['--source-8c', SOURCE_8C] if SOURCE_8C else []
    run('external-text-fixed-set', 'alignment_transfer_s2ut.evaluate_t2tt_external', 'prepare',
        '--benchmark', reference, '--output', fixed_set, *source_args)
    output = Path(ROOT) / 'external-text' / T2TT_RUN_ID / T2TT_TOKENIZER / 'report/text-bleu.json'
    run('external-text-bleu-' + T2TT_RUN_ID + '-' + T2TT_TOKENIZER, 'alignment_transfer_s2ut.evaluate_t2tt_external', 'evaluate',
        '--fixed-set', fixed_set, '--checkpoint', text_work / 'checkpoint_best.pt',
        '--output', output, '--device', 'cuda', '--resume', gpu=True)
    result = json.loads(output.read_text(encoding='utf-8'))
    print(json.dumps(result, ensure_ascii=False, indent=2))
else:
    log_event('external-text-bleu', 'skipped', LOGS, 'EVALUATE_T2TT_EXTERNAL=False')

## 2. 入力とBaselineを復元

完了済みの前処理を復元し、音声ZIPをVMに展開します。容量表示を確認してください。

In [ ]:
if EVALUATE_S2UT:
    corpus_common(restore_only=True)
    corpus_stage_audio()
    corpus_prepare(restore_only=True)
    corpus_baseline(restore_only=True)
else:
    log_event('restore-inputs', 'skipped', LOGS, 'S2UT evaluation disabled')


In [ ]:
if EVALUATE_S2UT:
    run('register-baseline', 'alignment_transfer_s2ut.benchmark', 'register',
        '--checkpoint', RESTORED / 'checkpoints/checkpoint_best.pt', '--data', PREPARED,
        '--recipe-lock', RESTORED / 's2ut-recipe-lock.json', '--metadata', RESTORED / 'research-metadata.json',
        '--output', BASELINE, '--resume')
    baseline = json.loads(BASELINE.read_text(encoding='utf-8'))
    print({key:baseline[key] for key in ('checkpoint_updates','training_budget','selection_policy')})
    BACKBONE_CONFIG.write_text(json.dumps({'backbone':baseline['backbone']}),encoding='utf-8')
else:
    log_event('registration', 'skipped', LOGS, 'S2UT evaluation disabled')


In [ ]:
if EVALUATE_S2UT:
    run('full-manifest', 'alignment_transfer_s2ut.benchmark_data', 'manifest',
        '--common', COMMON, '--prepared', PREPARED, '--output', MANIFEST, '--resume', restore_only=True)
    run('import-units', 'alignment_transfer_s2ut.benchmark_data', 'units', '--manifest', MANIFEST,
        '--benchmark-units', BENCHMARK_UNITS, '--prepared', PREPARED,
        '--output', Path(ROOT) / 'units', '--resume', restore_only=True)
else:
    log_event('restore-speech-artifacts', 'skipped', LOGS, 'S2UT evaluation disabled')


## 3. 比較するモデルを復元

In [ ]:
if EVALUATE_S2UT:
    RESEARCH_CONDITIONS = [(method,label,globals()['COMPARE_' + method.upper()]) for method,label in METHOD_LABELS]
    for method,label,enabled in RESEARCH_CONDITIONS:
        if enabled:
            execute_json(PYTHON, '-m', 'alignment_transfer_s2ut.colab_runtime', 'restore-training',
                '--work', str(RESEARCH_ROOT / method),
                '--backup', str(Path(PERSISTENT) / 'stages' / ('research-v3-' + method + SCOPE_SUFFIX)),
                '--method', method, name='restore-' + method)
        else:
            log_event(method, 'skipped', LOGS, 'condition excluded from evaluation')
else:
    log_event('restore-models', 'skipped', LOGS, 'S2UT evaluation disabled')


## 4. 共通推論とdev評価

UERは低いほど良く、ASR-BLEUは高いほど良い指標です。波形生成とASR-BLEUは設定で有効にします。

In [ ]:
if EVALUATE_S2UT:
    conditions = []
    from IPython.display import Markdown, display

    if not GENERATE_WAVEFORMS:
        log_event('waveform-generation', 'skipped', LOGS, 'GENERATE_WAVEFORMS=False')
    if not ASR_BLEU:
        log_event('asr-bleu', 'skipped', LOGS, 'ASR_BLEU=False')

    transfer_conditions = []
    for method, label, enabled in RESEARCH_CONDITIONS:
        folder = RESEARCH_ROOT / method
        if enabled:
            transfer_conditions.append((method, label, folder, RESEARCH_ROOT / 'registrations' / (method + '.json'), 'research-v3-register-' + method))
    missing = [label for _, label, folder, _, _ in transfer_conditions
               if not (folder / 'checkpoints/checkpoint_best.pt').is_file()]
    if missing:
        log_event('dev-comparison', 'skipped', LOGS, '学習完了待ち: ' + ', '.join(missing))
    else:
        conditions = [('baseline', 'Baseline', BASELINE)]
        for name, label, folder, record, registration in transfer_conditions:
            run(registration, 'alignment_transfer_s2ut.benchmark', 'register-transfer', '--run', folder,
                '--checkpoint', folder / 'checkpoints/checkpoint_best.pt', '--output', record, '--resume')
            conditions.append((name, label, record))
        records = [record for _, _, record in conditions]
        comparison_name = '-'.join(name for name, _, _ in conditions)
        COMPARISON_DIR = Path(ROOT) / 'comparison' / comparison_name
        if len(records) > 1:
            run(comparison_name + '-compare-training', 'alignment_transfer_s2ut.benchmark', 'compare', *records,
                '--output', COMPARISON_DIR / 'training-comparison.json', '--resume')
        source_only = Path(ROOT) / 'dev-source/dev-source.jsonl'
        run('dev-source', 'alignment_transfer_s2ut.prepare_inference_input', '--manifest', MANIFEST, '--split', 'dev', '--output', source_only, '--resume')
        if GENERATE_WAVEFORMS:
            run('vocoder', 'alignment_transfer_s2ut.fetch_assets', '--config', 'configs/s2ut/vocoder.yaml')
        evaluations = []
        table = ['| Condition | ASR-BLEU | UER |', '|---|---:|---:|']
        for name, label, record in conditions:
            predictions = Path(ROOT) / 'evaluation' / name / 'dev-predictions.jsonl'
            vocoder = ['--vocoder-config', 'configs/s2ut/vocoder.yaml'] if GENERATE_WAVEFORMS else ['--units-only']
            run(name + '-infer', 'alignment_transfer_s2ut.benchmark', 'infer', '--record', record, '--input', source_only,
                '--output', predictions, '--device', 'cuda', '--resume', *vocoder, gpu=True)
            metrics = predictions.with_name('dev-metrics.json')
            asr = ['--asr-config', 'configs/experiments/evaluation.yaml', '--device', 'cuda'] if ASR_BLEU else []
            run(name + '-evaluate', 'alignment_transfer_s2ut.evaluate', '--manifest', MANIFEST, '--split', 'dev',
                '--units', Path(ROOT) / 'units', '--predictions', predictions, '--output', metrics, '--resume', *asr, gpu=ASR_BLEU)
            evaluations.append(metrics)
            values = json.loads(metrics.read_text())
            bleu = f"{values['asr_bleu']:.2f}" if values.get('asr_bleu') is not None else '—'
            uer = f"{values['unit_error_rate']:.4f}" if values.get('unit_error_rate') is not None else '—'
            table.append(f'| {label} | {bleu} | {uer} |')
        if len(records) > 1:
            run(comparison_name + '-comparison', 'alignment_transfer_s2ut.benchmark', 'compare', *records, '--evaluations', *evaluations,
                '--output', COMPARISON_DIR / 'comparison.json', '--resume')
            print('比較レポート:', COMPARISON_DIR / 'comparison.json')
        display(Markdown('\n'.join(table)))
else:
    log_event('evaluate', 'skipped', LOGS, 'S2UT evaluation disabled')
